In [1]:
import boto3

Setting up the s3 client and retrieving available buckets

In [3]:
s3 = boto3.client('s3')

In [4]:
s3.list_buckets()

{'ResponseMetadata': {'RequestId': 'CTZ7RHGR901FTTX4',
  'HostId': 'K5CT2q8EjKRBKYqSpX0pteyRgU2jTpkz8JgW/8/THb9y6JPIcT5dAyD10X5wf8GknMMeELSAeKqYhQEgCM3ETkxJM2Bn0r/5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'K5CT2q8EjKRBKYqSpX0pteyRgU2jTpkz8JgW/8/THb9y6JPIcT5dAyD10X5wf8GknMMeELSAeKqYhQEgCM3ETkxJM2Bn0r/5',
   'x-amz-request-id': 'CTZ7RHGR901FTTX4',
   'date': 'Fri, 18 Jul 2025 08:47:28 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [],
 'Owner': {'ID': '0315aeb932d687bcf20c7d888d081736b8071cad76a537e5e331a023e821f737'}}

# Create a new Bucket

Provide a bucket name and a region to create a new bucket

In [6]:
bucket_name = 'soosisbucket'

In [ ]:
def create_bucket(bucket_name):i
    """Create an S3 bucket in a specified region"""
    try:
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={
            'LocationConstraint': "eu-north-1"
        })
        print(f"Bucket {bucket_name} created successfully.")
    except Exception as e:
        print(f"Error creating bucket: {e}")

In [9]:
create_bucket(bucket_name)

Bucket soosisbucket created successfully.


# Upload single file to S3

Selecting a file and uploading it

In [11]:
import os

def upload_file_to_s3(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket"""
    if object_name is None:
        object_name = os.path.basename(file_name)
    
    try:
        s3.upload_file(file_name, bucket, object_name)
        print(f"File {file_name} uploaded to {bucket}/{object_name}.")
    except Exception as e:
        print(f"Error uploading file: {e}")

In [15]:
upload_file_to_s3('boto_s3.ipynb', bucket_name)

File boto_s3.ipynb uploaded to soosisbucket/boto_s3.ipynb.


In [17]:
response = s3.list_objects_v2(Bucket=bucket_name)
for obj in response["Contents"]:
    print(obj["Key"])

boto_ec2.ipynb
boto_s3.ipynb


# Download S3 File to Local System

Downloading a single file from the S3 bucket

In [19]:
def download_file_from_s3(bucket_name, object_name, file_name):
    """Download a file from an S3 bucket"""
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    try:
        s3.download_file(bucket_name, object_name, file_name)
        print(f"File {object_name} downloaded from {bucket_name} to {file_name}.")
    except Exception as e:
        print(f"Error downloading file: {e}")

In [20]:
download_file_from_s3(bucket_name, 'boto_s3.ipynb', 'data_download/downloaded_boto_s3.ipynb')

File boto_s3.ipynb downloaded from soosisbucket to data_download/downloaded_boto_s3.ipynb.


# Upload Folder to S3

Uploading a whole folder to S3

In [ ]:
def upload_directory_to_s3(directory, bucket_name, s3_prefix):
    """Upload a directory to an S3 bucket"""
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            relative_path = os.path.relpath(file_path, directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(file_path, relative_path, s3_key)
            upload_file_to_s3(file_path, bucket_name, s3_key)

In [27]:
upload_directory_to_s3('data', bucket_name, "s3_data")

data/panda-1236875_640.jpg panda-1236875_640.jpg s3_data/panda-1236875_640.jpg
File data/panda-1236875_640.jpg uploaded to soosisbucket/s3_data/panda-1236875_640.jpg.
data/index.html index.html s3_data/index.html
File data/index.html uploaded to soosisbucket/s3_data/index.html.
data/water-5393919_1280.jpg water-5393919_1280.jpg s3_data/water-5393919_1280.jpg
File data/water-5393919_1280.jpg uploaded to soosisbucket/s3_data/water-5393919_1280.jpg.


# Download S3 Folder to Local System

Downloading a folder from S3 by iterating over every file available

In [33]:
local_path = 'data_download'
s3_prefix = 's3_data/'


def download_dir_from_s3(bucket_name, s3_prefix):
    """Download a directory from an S3 bucket"""
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    paginator = s3.get_paginator("list_objects_v2")
    for result in paginator.paginate(Bucket=bucket_name, Prefix=s3_prefix):
        if "Contents" in result:
            for obj in result["Contents"]:
                s3_key = obj["Key"]
                print(s3_key)
                local_file_path = os.path.join(local_path, s3_key.replace(s3_prefix, ""))
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

                s3.download_file(bucket_name, s3_key, local_file_path)
                print(f"Downloaded {obj['Key']} to {local_file_path}.")

In [32]:
download_dir_from_s3(bucket_name, s3_prefix)

s3_data/index.html
Downloaded s3_data/index.html to data_download/index.html.
s3_data/panda-1236875_640.jpg
Downloaded s3_data/panda-1236875_640.jpg to data_download/panda-1236875_640.jpg.
s3_data/water-5393919_1280.jpg
Downloaded s3_data/water-5393919_1280.jpg to data_download/water-5393919_1280.jpg.


# Delete all Files in an S3 Bucket

Delete all files to be able to delete the S3 bucket afterwards

In [ ]:
def delete_object_in_s3():
    """Delete all files in an S3 bucket"""
    paginator = s3.get_paginator("list_objects_v2")
    for result in paginator.paginate(Bucket=bucket_name):
        # print("Result: ", result)
        if "Contents" in result:
            for obj in result["Contents"]:
                s3.delete_object(Bucket=bucket_name, Key=obj["Key"])
                print(f"Deleted {obj['Key']} from {bucket_name}.")

In [35]:
delete_object_in_s3()

Result:  {'ResponseMetadata': {'RequestId': 'RFD11BCPP2K2R745', 'HostId': 'FWwH3Zh1hzYhRs7kOBISlT8eWYr/LZgki7XJ7I5u6tI7EIX3GYOVxzHrWKoKSXoFItnqZSn7bZ8=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'FWwH3Zh1hzYhRs7kOBISlT8eWYr/LZgki7XJ7I5u6tI7EIX3GYOVxzHrWKoKSXoFItnqZSn7bZ8=', 'x-amz-request-id': 'RFD11BCPP2K2R745', 'date': 'Fri, 18 Jul 2025 09:31:25 GMT', 'x-amz-bucket-region': 'eu-north-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'boto_ec2.ipynb', 'LastModified': datetime.datetime(2025, 7, 18, 8, 56, 36, tzinfo=tzutc()), 'ETag': '"0315bb7fd50b019dbb77ee25378efcb1"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': 'FULL_OBJECT', 'Size': 41985, 'StorageClass': 'STANDARD'}, {'Key': 'boto_s3.ipynb', 'LastModified': datetime.datetime(2025, 7, 18, 8, 57, 21, tzinfo=tzutc()), 'ETag': '"a59c6703840be3986ba5902ac760c9c4"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': '

In [ ]:
s3.delete_bucket(Bucket=bucket_name)
print(f"Bucket {bucket_name} deleted successfully.")

Bucket soosisbucket deleted successfully.
